In [ ]:
import pandas as pd
import json
from openai import OpenAI
from tqdm import tqdm
import time

In [ ]:
client=OpenAI(api_key)
file_path=r"C:\Git_for_study\DE\Work\Neg_data\sample_neg_df_260730(2).xlsx"
df=pd.read_excel(file_path)
print(f'총 {len(df)}건의 데이터를 불러왔습니다.')

총 780건의 데이터를 불러왔습니다.


In [5]:
df['AI_Label'] = None
df['AI_Confidence'] = None
df['AI_Reason'] = None

In [6]:
system_prompt = """
당신은 경기도 및 기초지자체의 도정/시정 뉴스를 모니터링하여 '부정적 위기(Negative Issue)' Alert 발송 유효성을 평가하는 AI 뉴스 분류기입니다.
제공된 기사 JSON 배열을 분석하여, 각 기사별 판단 결과를 JSON 배열로 반환하세요.

# [분류 기준 (Label: 1) - 부정/위기 뉴스]
아래 중 하나라도 해당하면 발송 대상(1)입니다.
1. 행정 비판 및 리스크: 경기도 또는 도내 지자체의 정책 실패, 예산 낭비, 비리, 행정 처분, 소송 등에 대한 비판적 보도
2. 지역 사회 재난/피해: 지역 주민의 안전이나 삶에 큰 영향을 미치는 대형 재난, 환경 오염, 대규모 집단 민원 및 시위
3. 지자체장 및 공직자 논란: 도지사, 시장, 공무원 등의 부정적 언행이나 법적/도덕적 구설수

# [제외 기준 (Label: 0) - 가비지/불요 뉴스]
아래 중 하나라도 해당하면 무조건 제외 대상(0)입니다.
1. 단순 사건·사고: 개인 간의 범죄, 일반적인 교통사고, 소규모 화재 등 지자체의 구조적 책임과 무관한 단순 경찰/소방 뉴스
2. 타 지역 및 국가 이슈: 경기도가 주체가 아닌, 중앙 정부나 타 지자체의 부정적 이슈에 경기도가 단순 언급만 된 경우
3. 민간 기업의 악재: 경기도 내 위치한 사기업의 주가 하락, 실적 부진, 제품 결함 등 (도정에 영향이 없는 경우)

# 확신도(Confidence) 기준
- 높음: 도정에 타격을 주는 명백한 비판/재난이거나, 명백한 단순 사건사고인 경우
- 중간/낮음: 개인의 범죄/사고이나 지자체의 관리 책임이 일부 섞여 있어 판단이 모호한 엣지 케이스

# 출력 형식: 반드시 아래와 같은 형태의 JSON 배열만 출력하세요.
[
  {"id": 0, "Label": 1, "Confidence": "높음", "Reason": "경기도 산하 공공기관의 채용 비리 의혹 보도로 핵심 부정 이슈임"},
  {"id": 1, "Label": 0, "Confidence": "중간", "Reason": "안양시 내 교통사고 보도로 지자체의 정책적 책임과 무관함"}
]
"""

In [7]:
batch_size = 10
total_batches = (len(df) // batch_size) + (1 if len(df) % batch_size > 0 else 0)
print(f"데이터를 {batch_size}개씩 묶어 총 {total_batches}번의 API 요청을 시작합니다...")
for i in tqdm(range(0, len(df), batch_size)):
    batch_df = df.iloc[i:i+batch_size]
    
    # 모델에 전달할 JSON 데이터 만들기
    input_data = []
    for idx, row in batch_df.iterrows():
        input_data.append({
            "id": idx,
            "title": str(row['Title']),
            # 비용 절감 및 토큰 제한을 위해 본문은 앞 300자만 사용
            "content": str(row['Content'])[:300] 
        })
    
    user_prompt = f"다음 기사들을 분석해주세요:\n{json.dumps(input_data, ensure_ascii=False)}"
    
    try:
        # GPT-4o-mini 모델 호출 (가장 저렴하고 빠름)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1
        )
        
        # 결과 파싱
        result_str = response.choices[0].message.content
        if "```json" in result_str:
            result_str = result_str.split("```json").split("```").strip()[1]
        elif "```" in result_str:
            result_str = result_str.split("```")[1].strip()
            
        result_json = json.loads(result_str)
        
        # 데이터프레임에 매핑
        for item in result_json:
            row_idx = item["id"]
            df.at[row_idx, 'AI_Label'] = item.get("Label")
            df.at[row_idx, 'AI_Confidence'] = item.get("Confidence")
            df.at[row_idx, 'AI_Reason'] = item.get("Reason")
            
    except Exception as e:
        print(f"\n[오류 발생] {i}번째 데이터 처리 중 에러: {e}")
        time.sleep(3) # 오류 시 잠시 대기 후 계속 진행
        continue

# ==========================================
# 5. 최종 결과 엑셀 저장
# ==========================================
output_path = "result_labeled_neg_df.xlsx"
df.to_excel(output_path, index=False)
print(f"\n라벨링 완료! '{output_path}' 파일이 생성되었습니다.")

데이터를 10개씩 묶어 총 78번의 API 요청을 시작합니다...


100%|██████████| 78/78 [07:03<00:00,  5.42s/it]


라벨링 완료! 'result_labeled_neg_df.xlsx' 파일이 생성되었습니다.
